# HaNoRec × LLM2Rec CF-hardness preflight

This kernel is a compatibility and contract preflight only. It does not train Qwen2.5-VL, run DPO, or claim a ranking result. It records whether Kaggle can see the frozen LLM2Rec/visual artifacts and whether the Phase 1 hardness formula is numerically well-defined.

Protocol: `plans/260918-1114-hanorec-cf-hardness/phase-01-local-spec-and-audit.md`.

In [ ]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
from pathlib import Path

ROOT = Path('/kaggle/working')
INPUT = Path('/kaggle/input')
ARTIFACT = ROOT / 'hanorec_cf_hardness_preflight.json'
SOURCE_PINS = {
    'llm2rec_commit': '73b481f710f67166ab958f4985d27b27fb410871',
    'hanorec_commit': '587face74524e4553b5a7aa295fe962004682382',
}
print({'python': sys.version, 'platform': platform.platform(), 'input': str(INPUT)})

## Block 1 — discover mounted artifacts

Kaggle mounts kernel outputs under slug-dependent directories. Discovery is filename-based; the preflight must not assume a slug path.

In [ ]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def discover(names: list[str]) -> dict[str, list[str]]:
    result = {}
    for name in names:
        result[name] = sorted(str(p) for p in INPUT.rglob(name)) if INPUT.exists() else []
    return result

expected_names = [
    'checkpoint500_title_embeddings.npy',
    'checkpoint1000_title_embeddings.npy',
    'item_embeddings.npy',
    'sasrec_scores.npy',
    'visual_features.npy',
    'availability.npy',
    'item_asin_map.json',
]
discovered = discover(expected_names)
print(json.dumps(discovered, indent=2))

## Block 2 — verify available files and hashes

Missing inputs are recorded as a blocked preflight condition. No substitute artifact is accepted.

In [ ]:
file_report = []
for name, paths in discovered.items():
    for raw_path in paths:
        path = Path(raw_path)
        file_report.append({'name': name, 'path': raw_path, 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
print(json.dumps(file_report, indent=2))
print(f'found_files={len(file_report)}')

## Block 3 — formula smoke test

This tests only the frozen Phase 1 formula: `lambda_combined = lambda_sem ** w * lambda_cf ** (1-w)`. It is not model training or evaluation.

In [ ]:
def sigmoid(x: float) -> float:
    return 1.0 / (1.0 + math.exp(-x))

def normalized_cf_hardness(margin: float, batch_mean: float) -> float:
    value = sigmoid(margin) / sigmoid(batch_mean)
    if not math.isfinite(value) or value <= 0.0:
        raise ValueError('CF hardness must be finite and positive')
    return value

lambda_sem = 0.73
lambda_cf = normalized_cf_hardness(0.05, 0.10)
if lambda_cf <= 0.0:
    raise AssertionError('lambda_cf must be positive')
values = {str(w): lambda_sem ** w * lambda_cf ** (1.0 - w) for w in (0.0, 0.5, 1.0)}
assert math.isclose(values['1.0'], lambda_sem)
assert math.isclose(values['0.0'], lambda_cf)
print(json.dumps({'lambda_sem': lambda_sem, 'lambda_cf': lambda_cf, 'lambda_combined': values}, indent=2))

## Block 4 — write auditable result

`READY_FOR_PHASE_2` is emitted only when all required artifacts are present. This run is therefore fail-closed.

In [ ]:
required = ['checkpoint500_title_embeddings.npy', 'sasrec_scores.npy', 'visual_features.npy', 'availability.npy', 'item_asin_map.json']
missing = [name for name in required if not discovered.get(name)]
status = 'READY_FOR_PHASE_2' if not missing else 'BLOCKED_MISSING_ARTIFACTS'
result = {
    'pipeline': 'hanorec-cf-hardness-preflight',
    'status': status,
    'missing': missing,
    'source_pins': SOURCE_PINS,
    'formula': 'lambda_sem ** w * lambda_cf ** (1-w)',
    'arms': [0.0, 0.5, 1.0],
    'discovered_files': file_report,
}
ARTIFACT.write_text(json.dumps(result, indent=2), encoding='utf-8')
print(json.dumps(result, indent=2))
if missing:
    print('Preflight intentionally stopped before training.')